# Notebook 0 — Nettoyage & Feature Engineering

> Chargement du CSV brut, correction des anomalies, agrégation en 64K profils clients × 25 features.

**Projet** : BDD #7 Sephora × Albert School | Groupe 5 | Case 2  
**Seed** : 42 (reproductibilité garantie)


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.append(os.path.join(os.getcwd(), '..'))  # accès au package src/

RANDOM_SEED = 42

---

## 0.1 Chargement du dataset brut

Lecture du CSV avec les bons types (IDs en string, pas float).


### Chargement CSV

Lecture avec pandas, vérification des types et du nombre de lignes attendu.


In [ ]:

from src.feature_engineering import load_raw_data, RAW_CSV
from src.utils import DATA_PATH
import os

# Chemin vers le CSV brut (à déposer dans data/)
csv_path = os.path.join(DATA_PATH, RAW_CSV)
print(f"Recherche : {csv_path}")
print(f"Existe    : {os.path.exists(csv_path)}")

df_raw = load_raw_data(csv_path)
print(f"\nShape : {df_raw.shape}")
print(f"Colonnes : {list(df_raw.columns)}")


### Vérification initiale

Shape, dtypes, valeurs manquantes, doublons.


In [ ]:

# Vérification initiale
print("=== SHAPE ===")
print(f"Lignes : {len(df_raw):,} | Colonnes : {df_raw.shape[1]}")
print(f"Clients uniques : {df_raw['anonymized_card_code'].nunique():,}")
print(f"Tickets uniques : {df_raw['anonymized_Ticket_ID'].nunique():,}")

print("\n=== TYPES ===")
print(df_raw.dtypes)

print("\n=== VALEURS MANQUANTES ===")
nulls = df_raw.isnull().sum()
nulls_pct = (nulls / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({"count": nulls, "pct": nulls_pct})
print(missing_df[missing_df["count"] > 0].sort_values("count", ascending=False))

print("\n=== PERIOD ===")
print(f"Date min : {df_raw['transactionDate'].min()}")
print(f"Date max : {df_raw['transactionDate'].max()}")

print("\n=== DOUBLONS ===")
dupes = df_raw.duplicated().sum()
print(f"Lignes dupliquées : {dupes}")


---

## 0.2 Correction des anomalies de qualité

Correction des 5 anomalies identifiées dans les données.


### Typo MAEK UP → MAKE UP

Remplacement dans Axe_Desc et Axe_Desc_first_purchase.


In [ ]:

# Anomalie 1 : Typo MAEK UP → MAKE UP
print("Avant correction :")
print(df_raw["Axe_Desc"].value_counts())

df_clean = df_raw.copy()
df_clean["Axe_Desc"] = df_clean["Axe_Desc"].str.strip().replace("MAEK UP", "MAKE UP")
if "Axe_Desc_first_purchase" in df_clean.columns:
    df_clean["Axe_Desc_first_purchase"] = (df_clean["Axe_Desc_first_purchase"]
                                            .str.strip().replace("MAEK UP", "MAKE UP"))

print("\nAprès correction :")
print(df_clean["Axe_Desc"].value_counts())
n_corrected = (df_raw["Axe_Desc"] == "MAEK UP").sum()
print(f"\n✓ {n_corrected:,} lignes corrigées ({n_corrected/len(df_raw)*100:.2f}%)")


### Conversion des dates

transactionDate en datetime (format MM/DD/YYYY).


In [ ]:

# Anomalie 2 : Conversion des dates (déjà fait dans load_raw_data)
# Vérification du résultat
print("Type transactionDate :", df_clean["transactionDate"].dtype)
print("Nulls après parse    :", df_clean["transactionDate"].isna().sum())

# Anomalie 3 : gender 99999 → NaN
if "gender" in df_clean.columns:
    n_99999 = (df_clean["gender"] == 99999).sum()
    df_clean["gender"] = df_clean["gender"].replace(99999, np.nan)
    print(f"\n✓ gender 99999 → NaN : {n_99999} lignes")

# Anomalie 4 : age_category vides → Unknown
for col in ["age_category", "age_generation"]:
    if col in df_clean.columns:
        n_miss = df_clean[col].isna().sum()
        df_clean[col] = df_clean[col].fillna("Unknown").replace("", "Unknown")
        print(f"✓ {col} : {n_miss:,} NaN → 'Unknown'")

print(f"\nShape après nettoyage dates/gender : {df_clean.shape}")


### Nettoyage colonnes numériques

salesVatEUR, discountEUR, quantity : coerce + fillna.


In [ ]:

# Anomalie 5 : Nettoyage colonnes numériques
for col in ["salesVatEUR", "discountEUR", "quantity"]:
    if col in df_clean.columns:
        n_neg = (pd.to_numeric(df_clean[col], errors="coerce") < 0).sum()
        n_nan = pd.to_numeric(df_clean[col], errors="coerce").isna().sum()
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").fillna(0).clip(lower=0)
        print(f"✓ {col:20s} : {n_nan} NaN → 0 | {n_neg} négatifs → 0")

# Net spend (CA après remise)
df_clean["net_spend"] = df_clean["salesVatEUR"] - df_clean["discountEUR"].fillna(0)

print(f"\n=== RÉSUMÉ NUMÉRIQUES ===")
print(df_clean[["salesVatEUR", "discountEUR", "quantity", "net_spend"]].describe().round(2))

# CA global et discount global
ca_total   = df_clean["salesVatEUR"].sum()
disc_total = df_clean["discountEUR"].sum()
print(f"\n✓ CA total    : {ca_total:,.0f} EUR")
print(f"✓ Discount    : {disc_total:,.0f} EUR ({disc_total/ca_total*100:.1f}%)")


---

## 0.3 Feature Engineering — Valeur & Fréquence (RFM)

Calcul des features RFM par client.


### RFM de base

recency_days, frequency (nb tickets), monetary (total_spend).


In [ ]:

from src.feature_engineering import compute_rfm, REFERENCE_DATE

print(f"Date de référence (recency) : {REFERENCE_DATE}")
rfm = compute_rfm(df_clean)
print(f"\nRFM shape : {rfm.shape}")
print(rfm[["recency_days", "frequency", "monetary"]].describe().round(2))

# Distribution RFM
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, label in zip(axes,
    ["recency_days", "frequency", "monetary"],
    ["Recency (jours)", "Fréquence (tickets)", "Monetary (€)"]):
    data = rfm[col].clip(upper=rfm[col].quantile(0.99))
    ax.hist(data, bins=50, color="#FF0066", alpha=0.8, edgecolor="white")
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel(label)
    ax.set_ylabel("Nb clients")
    ax.axvline(data.median(), color="#1A1A1A", ls="--", label=f"Médiane: {data.median():.0f}")
    ax.legend(fontsize=9)
fig.suptitle("Distribution RFM — 64K clients", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.savefig("../outputs/figures/0_rfm_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Sauvegardé : outputs/figures/0_rfm_distribution.png")


### Features dérivées

avg_basket, max_basket, discount_ratio, avg_items_per_basket.


In [ ]:

# avg_basket, max_basket, avg_items, discount_ratio sont déjà dans compute_rfm()
print("Features RFM complètes :")
print(rfm.describe().round(3))

# Corrélations RFM
fig, ax = plt.subplots(figsize=(7, 5))
corr = rfm[["recency_days", "frequency", "monetary",
            "avg_basket", "max_basket", "discount_ratio"]].corr()
im = ax.imshow(corr, cmap="RdPu", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
for i in range(len(corr)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center",
                fontsize=8, color="white" if abs(corr.iloc[i,j]) > 0.5 else "#1A1A1A")
ax.set_title("Corrélations features RFM", fontweight="bold")
fig.tight_layout()
plt.savefig("../outputs/figures/0_corr_rfm.png", dpi=150, bbox_inches="tight")
plt.show()


---

## 0.4 Feature Engineering — Comportement & Diversité

Entropie de marques, diversité des axes, régularité.


### Entropie de Shannon

brand_entropy, axe_entropy — mesure de diversification.


In [ ]:

from src.feature_engineering import compute_behavioral
from src.utils import shannon_entropy

# Exemple sur un client
sample_client = df_clean["anonymized_card_code"].iloc[0]
grp = df_clean[df_clean["anonymized_card_code"] == sample_client]
brand_ca = grp.groupby("brand")["salesVatEUR"].sum()
ent = shannon_entropy(brand_ca)
print(f"Exemple client {sample_client}")
print(f"  Marques : {len(brand_ca)} | Entropie Shannon : {ent:.3f}")
print(f"  Distrib CA par marque : {brand_ca.to_dict()}")

# Calcul pour tout le dataset (peut prendre quelques minutes)
print("\n[INFO] Calcul compute_behavioral() sur tout le dataset...")
beh = compute_behavioral(df_clean)
print(f"Shape : {beh.shape}")
print(beh[["unique_brands", "brand_entropy", "axe_entropy", "icb_score"]].describe().round(3))


### Loyauté marque

top_brand_share — part de la marque principale dans le CA.


In [ ]:

# top_brand_share : part du CA consacré à la marque principale
print("Distribution top_brand_share :")
print(beh["top_brand_share"].describe().round(3))

# Clients mono-marque (top_brand_share = 1.0) vs diversifiés
n_mono = (beh["top_brand_share"] == 1.0).sum()
print(f"\n• Clients mono-marque (100% CA = 1 marque) : {n_mono:,} ({n_mono/len(beh)*100:.1f}%)")
print(f"• Marques distinctes median : {beh['unique_brands'].median():.0f}")

# ICB Score distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(beh["icb_score"], bins=50, color="#FF0066", alpha=0.8, edgecolor="white")
ax.axvline(beh["icb_score"].mean(), color="#1A1A1A", ls="--",
           label=f"Moyenne : {beh['icb_score'].mean():.1f}")
ax.axvline(70, color="#990033", ls=":", alpha=0.8, label="Seuil ICB > 70 (Beauty Addict)")
ax.set_title("Distribution ICB Score — Indice de Curiosité Beauté", fontweight="bold")
ax.set_xlabel("ICB Score (0–100)")
ax.set_ylabel("Nb clients")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/figures/0_icb_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

n_icb70 = (beh["icb_score"] >= 70).sum()
print(f"\n• Clients ICB > 70 : {n_icb70:,} ({n_icb70/len(beh)*100:.1f}%)")


### Régularité

avg_days_between_purchases, purchase_regularity (std des intervalles).


In [ ]:

from src.feature_engineering import compute_temporal

print("[INFO] Calcul compute_temporal() sur tout le dataset...")
temp = compute_temporal(df_clean)
print(f"Shape : {temp.shape}")
print(temp.describe().round(2))

# Tenure distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in zip(axes,
    ["tenure_days", "avg_days_between_purchases"],
    ["Tenure (jours cliente)", "Jours entre achats (moyenne)"]):
    data = temp[col].clip(upper=temp[col].quantile(0.99))
    ax.hist(data, bins=50, color="#FF0066", alpha=0.8, edgecolor="white")
    ax.axvline(data.median(), color="#1A1A1A", ls="--",
               label=f"Médiane: {data.median():.0f}j")
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("Jours")
    ax.set_ylabel("Nb clients")
    ax.legend()
fig.tight_layout()
plt.savefig("../outputs/figures/0_tenure_regularity.png", dpi=150, bbox_inches="tight")
plt.show()


---

## 0.5 Feature Engineering — Canal, Marché, Temporal

Features omnicanal, positionnement marché, tendance.


### Canal

pct_estore, is_omnichannel.


In [ ]:

from src.feature_engineering import compute_channel

chan = compute_channel(df_clean)
print("Canal features :")
print(chan.describe().round(3))

print(f"\n• Clients omnicanaux (>= 2 canaux) : {chan['is_omnichannel'].sum():,} ({chan['is_omnichannel'].mean()*100:.1f}%)")
print(f"• % estore moyen/client : {chan['pct_estore'].mean()*100:.1f}%")
print(f"• Clients 100% estore   : {(chan['pct_estore'] == 1.0).sum():,}")


### Marché

pct_exclusive, pct_selective, pct_sephora.


In [ ]:

from src.feature_engineering import compute_market

mkt = compute_market(df_clean)
print("Marché features :")
print(mkt.describe().round(3))

# Distribution pct_exclusive
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, label in zip(axes,
    ["pct_exclusive", "pct_selective", "pct_sephora"],
    ["% CA Exclusive", "% CA Selective", "% CA Sephora"]):
    ax.hist(mkt[col], bins=40, color="#FF0066", alpha=0.8, edgecolor="white")
    ax.axvline(mkt[col].mean(), color="#1A1A1A", ls="--",
               label=f"Moy: {mkt[col].mean()*100:.0f}%")
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("Part de CA")
    ax.legend()
fig.suptitle("Positionnement marché par client", fontweight="bold")
fig.tight_layout()
plt.savefig("../outputs/figures/0_market_dist.png", dpi=150, bbox_inches="tight")
plt.show()


### Temporal

tenure_days, trend_spend_monthly (pente régression linéaire CA/mois).


In [ ]:

# Temporal déjà calculé ci-dessus dans temp
# Tendance mensuelle — résumé
print("Clients avec tendance positive (CA croissant) :",
      (temp["trend_spend_monthly"] > 0).sum(),
      f"({(temp['trend_spend_monthly'] > 0).mean()*100:.1f}%)")
print("Clients avec tendance négative (CA décroissant) :",
      (temp["trend_spend_monthly"] < 0).sum(),
      f"({(temp['trend_spend_monthly'] < 0).mean()*100:.1f}%)")

fig, ax = plt.subplots(figsize=(8, 4))
trend_clip = temp["trend_spend_monthly"].clip(-200, 200)
ax.hist(trend_clip, bins=60, color="#FF0066", alpha=0.8, edgecolor="white")
ax.axvline(0, color="#1A1A1A", ls="--", lw=2, label="Stable")
ax.set_title("Distribution tendance CA mensuel (€/mois)", fontweight="bold")
ax.set_xlabel("Pente CA/mois (EUR)")
ax.set_ylabel("Nb clients")
ax.legend()
plt.tight_layout()
plt.show()


---

## 0.6 Split temporel & Sauvegarde

Division Jan–Juin / Juil–Sep et export des fichiers.


### Split temporel

customer_features_train.csv (Jan–Juin) et customer_features_test.csv (Juil–Sep).


In [ ]:

from src.feature_engineering import SPLIT_DATE

print(f"Split temporel : Train < {SPLIT_DATE.date()} | Test >= {SPLIT_DATE.date()}")

# Transactions train (Jan–Juin)
df_train_raw = df_clean[df_clean["transactionDate"] < SPLIT_DATE].copy()
df_test_raw  = df_clean[df_clean["transactionDate"] >= SPLIT_DATE].copy()

print(f"\nTrain (Jan–Juin) : {len(df_train_raw):,} lignes | "
      f"{df_train_raw['anonymized_card_code'].nunique():,} clients")
print(f"Test  (Juil–Sep) : {len(df_test_raw):,} lignes  | "
      f"{df_test_raw['anonymized_card_code'].nunique():,} clients")

# Overlap clients
train_clients = set(df_train_raw["anonymized_card_code"].unique())
test_clients  = set(df_test_raw["anonymized_card_code"].unique())
overlap = len(train_clients & test_clients)
print(f"\nClients dans les 2 périodes : {overlap:,} ({overlap/len(train_clients)*100:.1f}% du train)")
print("→ Ce sont ces clients pour lesquels on pourra détecter des migrations")


### Export final

Sauvegarde de customer_features.csv complet dans data/.


In [ ]:

from src.feature_engineering import build_feature_store, save_features

# Feature store complet sur toute la période
print("=== FEATURE STORE COMPLET (toute période) ===")
features_full = build_feature_store(df_clean)

# Feature store train uniquement (pour entraîner le clustering)
print("\n=== FEATURE STORE TRAIN (Jan–Juin) ===")
features_train = build_feature_store(df_train_raw)

# Sauvegarde
import os
os.makedirs("../data", exist_ok=True)
features_full.to_csv("../data/customer_features.csv")
features_train.to_csv("../data/customer_features_train.csv")
df_test_raw.to_csv("../data/transactions_test.csv", index=False)

print("\n✓ Fichiers sauvegardés :")
print("  → data/customer_features.csv")
print("  → data/customer_features_train.csv")
print("  → data/transactions_test.csv")
print(f"\nFEATURE STORE FINAL : {features_full.shape[0]:,} clients × {features_full.shape[1]} features")
print("\nAperçu colonnes :")
print(features_full.dtypes)
